# Grid of nearest framework atoms

In this notebook, along with the distance grid, we will also return the indices of the framework atom that is responsible for the shortest distance obtained in the distance grid.This is quite useful for finding the atoms around a given pocket or pockets as shown later in the tutorial. THis calculation will be twice as time consuming as the no cell list distance grid calculation.

## Imports

In [ ]:
import mofography as mgr
import numpy as np 
from pathlib import Path
from ase.io import read
import stackprinter 
import plotly.graph_objects as go
import plotly.express as px
stackprinter.set_excepthook(style='darkbg2')
path_to_cif = Path('../examples/TestCIFs/LTA96.cif')

print('Path exists:', path_to_cif.exists())



## Define function
Define a function to return the distance grid and the indices of the framework atoms that are responsible for the shortest distance to each grid point.


In [ ]:
def dgrid_and_frameindex_from_atoms_cpu_no_aabb(ase_atoms, radii=None, spacing=0.2, block_size=20000):
    """
    Similar to dgrid_from_atoms_cpu_no_aabb but returns both the distance grid and the indices
    of the closest framework atoms for each grid point.

    :param ase_atoms: ASE Atoms object containing the framework structure
    :param radii: Array of atomic radii for each atom. If None, uses default vdW radii
    :param spacing: Grid spacing in Angstroms, defaults to 0.2
    :param block_size: Size of blocks for parallel processing, defaults to 20000
    :return: Tuple of (distance_grid, argmin_grid) where:
            - distance_grid is a 3D array of minimum distances to framework surface
            - argmin_grid is a 3D array of indices of closest framework atoms
    """
     # data    = read(path_to_cif)

    
    from ase.io import read
    import numpy as np
    import dask.array as da
    import pandas as pd
    from tqdm import tqdm
    import ase
    fpoints = ase_atoms.get_scaled_positions() - 0.5
    cell = (ase.geometry.complete_cell(ase_atoms.get_cell())).T
    if radii is None:
         vdw_radii = pd.Series(
            {
                "H": 110.00000000000001,
                "He": 140.0,
                "Li": 182.0,
                "Be": 153.0,
                "B": 192.0,
                "C": 170.0,
                "N": 155.0,
                "O": 152.0,
                "F": 147.0,
                "Ne": 154.0,
                "Na": 227.0,
                "Mg": 173.0,
                "Al": 184.0,
                "Si": 210.0,
                "P": 180.0,
                "S": 180.0,
                "Cl": 175.0,
                "Ar": 188.0,
                "K": 275.0,
                "Ca": 231.0,
                "Sc": 215.0,
                "Ti": 211.0,
                "V": 206.99999999999997,
                "Cr": 206.0,
                "Mn": 204.99999999999997,
                "Fe": 204.0,
                "Co": 200.0,
                "Ni": 197.0,
                "Cu": 196.0,
                "Zn": 200.99999999999997,
                "Ga": 187.0,
                "Ge": 211.0,
                "As": 185.0,
                "Se": 190.0,
                "Br": 185.0,
                "Kr": 202.0,
                "Rb": 303.0,
                "Sr": 249.00000000000003,
                "Y": 231.99999999999997,
                "Zr": 223.0,
                "Nb": 218.00000000000003,
                "Mo": 217.0,
                "Tc": 216.0,
                "Ru": 213.0,
                "Rh": 210.0,
                "Pd": 210.0,
                "Ag": 211.0,
                "Cd": 218.00000000000003,
                "In": 193.0,
                "Sn": 217.0,
                "Sb": 206.0,
                "Te": 206.0,
                "I": 198.0,
                "Xe": 216.0,
                "Cs": 343.0,
                "Ba": 268.0,
                "La": 243.00000000000003,
                "Ce": 242.0,
                "Pr": 240.0,
                "Nd": 239.0,
                "Pm": 238.0,
                "Sm": 236.0,
                "Eu": 235.0,
                "Gd": 234.0,
                "Tb": 233.0,
                "Dy": 231.0,
                "Ho": 229.99999999999997,
                "Er": 229.0,
                "Tm": 227.0,
                "Yb": 225.99999999999997,
                "Lu": 224.00000000000003,
                "Hf": 223.0,
                "Ta": 222.00000000000003,
                "W": 218.00000000000003,
                "Re": 216.0,
                "Os": 216.0,
                "Ir": 213.0,
                "Pt": 213.0,
                "Au": 214.0,
                "Hg": 223.0,
                "Tl": 196.0,
                "Pb": 202.0,
                "Bi": 206.99999999999997,
                "Po": 197.0,
                "At": 202.0,
                "Rn": 220.00000000000003,
                "Fr": 348.0,
                "Ra": 283.0,
                "Ac": 247.00000000000003,
                "Th": 245.00000000000003,
                "Pa": 243.00000000000003,
                "U": 241.0,
                "Np": 239.0,
                "Pu": 243.00000000000003,
                "Am": 244.0,
                "Cm": 245.00000000000003,
                "Bk": 244.0,
                "Cf": 245.00000000000003,
                "Es": 245.00000000000003,
                "Fm": 245.00000000000003,
                "Md": 246.0,
                "No": 246.0,
                "Lr": 246.0,
                "Rf": np.nan,
                "Db": np.nan,
                "Sg": np.nan,
                "Bh": np.nan,
                "Hs": np.nan,
                "Mt": np.nan,
                "Ds": np.nan,
                "Rg": np.nan,
                "Cn": np.nan,
                "Nh": np.nan,
                "Fl": np.nan,
                "Mc": np.nan,
                "Lv": np.nan,
                "Ts": np.nan,
                "Og": np.nan,})
    radii = (vdw_radii[ase_atoms.get_chemical_symbols()].values / 100.0).reshape(-1, 1)

    fpoints = ase_atoms.get_scaled_positions() - 0.5
    cell = (ase.geometry.complete_cell(ase_atoms.get_cell())).T

    [nx, ny, nz] = (ase_atoms.cell.cellpar()[0:3] / spacing).astype(int) + 1
    gpoints = (da.stack(
        da.meshgrid(
            np.linspace(-0.5, 0.5, nx),
            np.linspace(-0.5, 0.5, ny),
            np.linspace(-0.5, 0.5, nz),
            indexing="ij",
        ),
        -1,
    ).reshape(-1, 3).rechunk(block_size, 3))

    def gpd_np_with_argmin(points, fpoints=fpoints, cell=cell, radii=radii):
        diff = np.expand_dims(points, axis=1) - np.expand_dims(fpoints, axis=0)
        diff = (diff - np.around(diff)).reshape(-1, 3)
        diff = np.dot(cell, diff.T).T
        distances = np.linalg.norm(diff, axis=1).reshape(-1, fpoints.shape[0]) - radii.T
        min_distances = np.min(distances, axis=1)
        argmin_indices = np.argmin(distances, axis=1)
        return np.column_stack((min_distances, argmin_indices))

    # Map the function across blocks and compute both grids
    combined = gpoints.map_blocks(
        gpd_np_with_argmin,
        chunks=(block_size, 2),
        dtype=np.float32,
    ).compute()

    # Separate the results into distance grid and argmin grid
    dgrid = combined[:, 0].reshape(nx, ny, nz)
    argmin_grid = combined[:, 1].astype(np.int32).reshape(nx, ny, nz)

    return dgrid, argmin_grid

## Compute distance grid and frameindices

In [ ]:
mat_atoms = read(path_to_cif)


In [ ]:
spacing = 0.5
dgrid, frameindex_grid = dgrid_and_frameindex_from_atoms_cpu_no_aabb(mat_atoms, spacing=spacing, block_size=5000)
dgrid = mgr.spectral_smooth_3d(dgrid, mat_atoms, spacing=spacing)
regions, maxima = mgr.regions_from_dgrid(dgrid, mask_thickness=0.5, h=0.4)
maxima_radii = dgrid[tuple(maxima.T)]

## Plot the grid and regions

In [ ]:
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
frame_trace = mgr.plot_framework_as_mesh(mat_updated)
cell_trace = mgr.plot_cell(mat_atoms, color='purple')
isosurface_trace = mgr.plot_surface_dgrid(dgrid, isovalue=2, ase_atoms=mat_atoms, region_labels=regions, stylize=True, select='all', opacity=1, cscale=None, color='blue', stepsize=1, hover_label='Region :')



In [ ]:
fig = go.Figure(data=[*frame_trace, cell_trace, isosurface_trace], layout=mgr.dark_layout)
fig.show(renderer='png')


## Pick region and atoms around it
- Pick a region at random, 
- Pick the set of atoms around all the points in that region

In [ ]:
interested_atoms_list = np.unique(frameindex_grid[regions==14])

In [ ]:

frame_subset_trace = mgr.plot_subset_of_atoms(mat_atoms, interested_atoms_list)


In [ ]:

fig = go.Figure(data=[*frame_subset_trace, cell_trace, isosurface_trace], layout=mgr.dark_layout)
fig.show()


>NOTE: You might see some periodic atom images in the final plot, these are added to show the bonds more clearly, and are not included in the list pulled from the frameindices grid. 

## Grab chemical nodes and linkers for MOF
So now we got the atom indices, but, what if we want the whole chemical nodes and linkers that contain these atoms? We can use mofid to breakdown the MOF into building blocks and then just grab the corresponding chemical nodes and linkers as a whole.

First let's repeat all above steps for an actual MOF instead of a zeolite. 

In [ ]:
path_to_cif = Path('../examples/TestCIFs/P1-Cu-BTC.cif')
mat_atoms=read(path_to_cif)
spacing = 0.5
dgrid, frameindex_grid = dgrid_and_frameindex_from_atoms_cpu_no_aabb(mat_atoms, spacing=spacing, block_size=5000)
dgrid = mgr.spectral_smooth_3d(dgrid, mat_atoms, spacing=spacing)
regions, maxima = mgr.regions_from_dgrid(dgrid, mask_thickness=0.5, h=0.4)
maxima_radii = dgrid[tuple(maxima.T)]
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
frame_trace = mgr.plot_framework_as_mesh(mat_updated)
cell_trace = mgr.plot_cell(mat_atoms, color='purple')
isosurface_trace = mgr.plot_surface_dgrid(dgrid, isovalue=2, ase_atoms=mat_atoms, region_labels=regions, stylize=True, select='all', opacity=1, cscale=None, color='blue', stepsize=1, hover_label='Region :')
fig = go.Figure(data=[*frame_trace, cell_trace, isosurface_trace], layout=mgr.dark_layout)
fig.show(renderer='png')


In [ ]:

interested_atoms_list = np.unique(frameindex_grid[regions==18])
frame_subset_trace = mgr.plot_subset_of_atoms(mat_atoms, interested_atoms_list)
fig = go.Figure(data=[*frame_subset_trace, cell_trace, isosurface_trace], layout=mgr.dark_layout)
fig.show(renderer='png')


That looks Ok so far. Feel free to play around with picking different regions. Like before, you might see some periodic atom images in the final plot, these are added to show the bonds more clearly, and are not included in the list pulled from the grid. 

### Breakdown MOF using MOFId
I am using my own fork of mofid for Windows which is updated to latest gcc standards : https://github.com/arung-northwestern/mofid-cygwin/tree/master it's public. If you're on linux, you're probably better off using https://github.com/snurr-group/mofid, so modify these commands according to their docs

In [ ]:
from mofid.id_constructor import make_command, run_cygwin_from_jupyter_rt, extract_fragments, extract_topology
from mofid.run_mofid import cif2mofid

path_to_mofid = Path('C:/Users/arung/OneDrive - The University of Manchester/my_cloned_repos/mofid_clones/mofid-cygwin/')

Let's breakdown the MOF to chemical nodes and linkers. 
>NOTE: Node smiles detection doesn't work that well sometimes. We will find a work around to this in the near future. 
>NOTE: If it doesn't any building blocks at all, try relaxing the structure to make the bondlengths more proper using UFF4MOF or just UFF, then try again. 

In [ ]:
breakdown= cif2mofid(cif_path=path_to_cif, output_path=Path('.').resolve(), path_to_mofid=path_to_mofid)

In [ ]:
breakdown['smiles_nodes'], breakdown['smiles_linkers']

### Make atoms and structure graphs
We need to make atoms objects and structure grpahs for the linkers, nodes and MOF for future use. For nodes, it might throw and error if the smiles is not accurate. which is ok, for now.

In [ ]:

node_atoms = mgr.get_atoms_from_smiles(breakdown['smiles_nodes'][0])
linker_atoms = mgr.get_atoms_from_smiles(breakdown['smiles_linkers'][0])
linker_sgraph = mgr.sgraph_from_atoms(linker_atoms)
mof_sgraph=mgr.sgraph_from_atoms(mat_atoms)

In [ ]:
breakdown['smiles_linkers']

### Visualize linker atoms

In [ ]:
fig = go.Figure(data=mgr.plot_framework_as_mesh(linker_atoms), layout=mgr.dark_layout).show(renderer='png')

Looks good enough, the linker is detecte correctly by MOFId.

### Find linkers in the framework
We can match a given pattern (subgraph) to a larger graph, with additional criteria on nodes and/or linkers using subgraph isomorphism algorithms in networkx. Check out the functional groups tutorial https://github.com/Shi-Research-Group/mofography/blob/master/docs/detect_functional_groups_in_dataset.ipynb for additional example of the same.

Let's use subgraph isomorphism to find the indices aof all the linker atoms in the MOF like this: 

In [ ]:
from networkx.algorithms import isomorphism
node_matcher = isomorphism.categorical_node_match('element', 'C')
matcher = isomorphism.GraphMatcher(mof_sgraph, linker_sgraph, node_match=node_matcher)
isomorphic_subgraphs = list(matcher.subgraph_isomorphisms_iter())

In theory, you could match the edge bond_type or bond_order as well, but, if the framework distances are not spot on, the bond_order calculated might be wrong, and you might miss matches. So that's not included here. We are just matching groups of connected element types only.

In [ ]:
frame_linkers_indices = [list(f.keys()) for f in isomorphic_subgraphs]
print(frame_linkers_indices), print(len(frame_linkers_indices))

# remove duplicate sublists in the list of sublists
# If you need to preserve order within the output sublists:
def get_unique_sublists_unordered_sorted(list_of_lists):
    unique_sets = set(frozenset(sublist) for sublist in list_of_lists)
    return [sorted(list(s)) for s in unique_sets]
unique_linkers = get_unique_sublists_unordered_sorted(frame_linkers_indices)
print(unique_linkers), print(len(unique_linkers))

> NOTE: You can see that now the actual number of matches is drastically reduced from 1536 to 32, this is because isomorphism add a separate entry based on the starting point for cycles in the linker, and CuBTC linker has many such cycles.

Let's verify if we are matching things correctly, by visualizing one set of the matched indices

In [ ]:
first_match_indices=unique_linkers[0]
subset_trace = mgr.plot_subset_of_atoms(mat_atoms, subset=first_match_indices)
go.Figure(data=[*subset_trace], layout=mgr.dark_layout).show(renderer='png')

Great! now all we need to do is to grab the sublists, where at least one of the nearest atoms from the previous search, is in that sublist

In [ ]:
selected_linker_indices = np.unique([uli for i in interested_atoms_list for uli, ul in  enumerate(unique_linkers) if i in ul])
selected_linkers = [unique_linkers[sli] for sli in selected_linker_indices]
print(selected_linkers), print(len(selected_linkers)) 



Great! we found the exact 8 linkers surrounding our region of interest. 

## Final visualization

In [ ]:
linkers_trace = mgr.plot_subset_of_atoms(mat_atoms, subset=np.hstack(selected_linkers))
fig = go.Figure(data=[cell_trace, isosurface_trace, *linkers_trace], layout=mgr.dark_layout)
fig.show(renderer='png')


## Consolidated Wrapper
Add a consolidated wrapper function that takes dgrid, frameindices and regions, mat_atoms, region of interest as input and returns : all the linkers, topology code and a list of sublists where each sublist contains the indices of the linkers that surround the regions of interest.

>> NOTE: Once you know the indices of all the linkers, you can make a subgraph of the rest and do connected components to get the individual chemical nodes subgraphs?  Assuming these chemical nodes are not touching each other like in some topologies.

## Conclusions
So here's what we did in this document: 
- Grabbed a cif file
- Compute the distance and frame indices grid
- Picked a region
- Found some atoms that are directly responsible for the distances in that region
- Decomposed the MOF into building blocks using MOFId
- Found matching patterns (linkers) in the framework using subgraph isomorphism
- Found the indices of the linkers that include the atoms we identfied earlier. 
- Visualized the said linkers only, around our region of interest.
